#  Objective - This code is built for prediction of the returning shipment

#  Description - The code will have basic 4 sections :
##                         1. Data Import and basic Quality Checks
##                         2. Exploratoy data analysis, adding features to data and making model ready data
##                         3. Building Random Forrest and Neural Network Models   
##                         4. Prediction of return rate using the validation data


# Date -            07 July 2021
# Author -        Avinash Mishra 


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import ta
import warnings
import time 
from IPython.display import clear_output
#from pyti.exponential_moving_average import exponential_moving_average as EMA
import xlrd 
import xlsxwriter
import openpyxl
import sys
import decimal
import pyodbc 
import pandas.io.sql
import smtplib
import mimetypes
import time
import datetime
import os
#import paho.mqtt.client as mqtt
import sys  
import csv
import socket
from datetime import datetime
from scipy.signal import argrelextrema
from scipy.signal import find_peaks, peak_prominences
import pickle
import matplotlib
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib import style
from functools import reduce 
import re
import seaborn as sns
import sklearn
from sklearn import preprocessing
from scipy.spatial.distance import euclidean
import glob
from datetime import date, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from IPython.display import clear_output
from numpy import arange
import os 
from sklearn.metrics import r2_score
import ta
from ta import *
import talib
#from rfpimp import permutation_importances

### Section  - 1 - Basic Data Quality Checks

In [ ]:
path = "<local-path>"
df= pd.read_csv(path + "TrainingData_V1.csv")


df.head()

In [ ]:
#### create some derived varibale in the data 
df['delivery_date']=pd.to_datetime(df['delivery_date'],format = '%d-%m-%Y')
df['order_date']=pd.to_datetime(df['order_date'],format = '%d-%m-%Y')
df['user_reg_date']=pd.to_datetime(df['user_reg_date'],format = '%d-%m-%Y')
df['user_dob']=pd.to_datetime(df['user_dob'],format = '%d-%m-%Y')
df['Today'] =pd.to_datetime(date.today(),format = '%Y-%m-%d')


df['Delivery_Time'] = (df['delivery_date'] -df['order_date'])// timedelta(days=1)
df['User_Age_years'] = (df['Today'] - df['user_dob'])// timedelta(days=365.2425)
df['System_Age_months'] =(df['Today'] - df['user_reg_date'])// timedelta(days=30)



In [ ]:
df['User_Age_years'].describe()

In [ ]:
df['Delivery_Time'] = np.where(df['Delivery_Time']< 0, df['Delivery_Time'].mean(),df['Delivery_Time'] )
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())
df['Delivery_Time'].describe()

In [ ]:
### get the columns with na values 
df[df.columns[df.isnull().any()]].isnull().sum() * 100 / df.shape[0]


# Section  2 : Exploratory Analysis to understand the data 

In [ ]:
df =df.sort_values(by=['order_date'], ascending=True)
df= df.reset_index()
df

###### Adding some derived variables in the data

#### Creating a variable for the user return history

In [ ]:



#df['User_level_return_count'] = df.groupby(by=['user_id','order_date'])['return'].sum()

#### Creating historical return history at a user level 

test = pd.DataFrame(df.groupby(by=['user_id','order_date'])['return'].sum())
test =test.reset_index()
test.columns = ['user_id','order_date','return']
test['user_return_history'] = test.groupby(['user_id'])['return'].transform(pd.Series.cumsum)
test['user_return_history'] =test.groupby('user_id')['user_return_history'].shift().fillna(0)
test=test[['order_date','user_id','user_return_history']]

df = df.merge(test, how='inner', on=['order_date','user_id'])


## Recent historical return rates by item,brand,size,color and state

In [ ]:
####### Recent historical return at an item and brand level 

#### list of variable to create return trend 

var = ['brand_id','item_id','item_size','user_state','item_color']

for each in var:


    test = pd.DataFrame()
    test =df.groupby(by=[each,'order_date']).agg(Total_Returns=('return', 'sum'),Total_Orders=('return', 'count'))
    test['Return_per'] = test['Total_Returns']/test['Total_Orders']
    test[str(each) + "_return_trend"] = test['Return_per'].rolling(3).mean()
    test[str(each) + "_return_trend"] =test.groupby(by =[each])[str(each) + "_return_trend"].shift().fillna(0)

    test = test.reset_index()
    test = test[['order_date',each,str(each) + "_return_trend"]]

    df = df.merge(test, how='inner', on=['order_date',each])


In [ ]:
df

## Getting return rates by various variables to analysed the differentiating variables

In [ ]:
col = ['order_date', 'item_id', 'item_size',
       'item_color', 'brand_id', 'item_price', 'user_id', 'user_title',
       'user_state', 'Delivery_Time',
        'System_Age_months', 'User_Age_years','user_return_history','brand_id_return_trend','item_id_return_trend','item_size_return_trend']

In [ ]:

with pd.ExcelWriter(path + 'Return_rates_groups_20210704v1.xlsx') as writer:
    
    for each in col:

        print(each)

        test = pd.DataFrame

        test = df.groupby([each]).agg(Total_Returns=('return', 'sum'),Total_Orders=('return', 'count'))
        test['Return_per'] = test['Total_Returns']/test['Total_Orders']
        test.to_excel(writer, sheet_name= each)

In [ ]:
#bins = [0, 20, 40,60,80, 100]
#df['binned_age'] = np.searchsorted(bins, df['User_Age_years'].values)
#df['binned_System_Age'] = np.searchsorted(bins, df['System_Age_months'].values)
#df['binned_Delivery_Time'] = np.searchsorted(bins, df['Delivery_Time'].values)
#df['binned_item_price'] = np.searchsorted(bins, df['item_price'].values)


    
    

# Section 3 - Building the models 

### Creating dummy variables for the categorical variables in the data

In [ ]:
##### Get the return type variable in the 

df_updated =pd.get_dummies(df, columns=["item_id","item_size","item_color","brand_id","user_title","user_state"])

In [ ]:
df_updated ['Price'] = df_updated ['item_price'] 

In [ ]:
df_updated = df_updated.dropna()
df_updated = df_updated.reset_index()
df_updated.iloc[:,11:]

#### Scaling the data for the model 

In [ ]:

sc = StandardScaler()
X =  pd.DataFrame(sc.fit_transform(df_updated.iloc[:,11:]))

In [ ]:
X.columns =df_updated.iloc[:,11:].columns
X

In [ ]:
df_updated.iloc[:,11:]

#### Splitting the data into test and training set

In [ ]:
#df_updated.columns[12:]

#df_updated = df_updated.dropna()
# create training and testing vars
X_train, X_test, y_train, y_test = train_test_split(X, df_updated['return'], test_size=0.3)


### Building the random forrest models using hyper parameter optmization

In [ ]:
import random
random.seed(1213) 


#name = currency_name+"_"+condition+"_"+tf

param_grid = { 
'n_estimators': [500],
'max_features': [ 'sqrt'],
'max_depth':[15],
'min_samples_leaf' :[2]  
}

rfc = RandomForestClassifier()

CV_rfc = GridSearchCV(estimator=rfc, param_grid=param_grid, cv= 5)
model =CV_rfc.fit(X_train, y_train)
best_params = pd.DataFrame()
best_params = CV_rfc.best_params_


#best_params

best_params

In [ ]:
rfc_predict = pd.DataFrame(model.predict_proba(X_test))
rfc_predict

In [ ]:
y_test =y_test.reset_index()

final_pred = pd.concat([rfc_predict,y_test],axis =1)
final_pred

In [ ]:
final_pred.columns =['Prob_0','Prob_1','index','return']

# Creating the predcited column in the data using probablity cut off

In [ ]:
final_pred['Predicted'] = np.where(final_pred['Prob_1']>=0.51,1,0)

In [ ]:
final_pred['Prob_1'].describe()

In [ ]:
Accuracy_table =final_pred.groupby(['return']).agg(Predicted_mean=('Predicted', 'mean'),Predicted_1=('Predicted', 'sum'),Predicted_0=('Predicted', 'count'))
Accuracy_table= Accuracy_table.reset_index()
Accuracy_table.columns = ['Actual_return','predicted_mean','predicted_1','predicted_all']
Accuracy_table['predicted_0'] = Accuracy_table['predicted_all'] - Accuracy_table['predicted_1']
Accuracy = (Accuracy_table['predicted_0'][0] + Accuracy_table['predicted_1'][1] )/Accuracy_table['predicted_all'].sum()
Accuracy

### Creating the confusion matrix table to asses the model perfromance and create lift chart

In [ ]:

p_table =pd.DataFrame()

list_a = [10,20,30,40,50,60,70,80,90]

i=0

for n in list_a:
           

            p_table.loc[i,'Percentile'] = n

            p_table.loc[i,'Cut_Off_Probability'] = np.percentile(np.array(final_pred['Prob_1']),n)

            cut_off = p_table.loc[i,'Cut_Off_Probability']

            p_table.loc[i,'TN'] = len(final_pred[final_pred['return'] == 0]\
                                      [final_pred['Prob_1'] < cut_off])

            p_table.loc[i,'FP'] = len(final_pred[final_pred['return'] == 0]\
                                      [final_pred['Prob_1'] >= cut_off])

            p_table.loc[i,'FN'] = len(final_pred[final_pred['return'] == 1]\
                                      [final_pred['Prob_1'] < cut_off])

            p_table.loc[i,'TP'] = len(final_pred[final_pred['return'] == 1]\
                                      [final_pred['Prob_1'] >= cut_off])

            p_table.loc[i,'Total'] = p_table.loc[i,'TN'] + p_table.loc[i,'FP'] + p_table.loc[i,'FN'] + p_table.loc[i,'TP']

            p_table.loc[i,'FP+TP'] = p_table.loc[i,'FP'] + p_table.loc[i,'TP']

        

            p_table.loc[i,'Predicted 1 Accuracy'] = p_table.loc[i,'TP']/(p_table.loc[i,'TP'] + p_table.loc[i,'FP'])

          

            i+=1



p_table.to_csv("Lift_chart_RF_0706V1.csv")


#### Getting the relative importance of variables in the models 

In [ ]:
from sklearn.metrics import r2_score
from rfpimp import permutation_importances

def r2(model, X_train, y_train):
    return r2_score(y_train, model.predict(X_train))

perm_imp_rfpimp = permutation_importances(model, X_train, y_train, r2)

In [ ]:
perm_imp_rfpimp.to_csv('relative_importance_v1.csv')

###  Building the Neural Network models 

In [ ]:
import random
random.seed(1) 


import tensorflow
tensorflow.random.set_seed(3)

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from IPython.display import clear_output
from numpy import arange
import datetime as dt
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, train_test_split
import os
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix


from sklearn.model_selection import train_test_split
from sklearn import model_selection
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix


from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from keras import Sequential
from keras.layers import Dense
import pickle
from sklearn.preprocessing import StandardScaler

from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import GridSearchCV


from keras.layers import Dense, Activation, Embedding, Flatten, LeakyReLU, BatchNormalization, Dropout
from keras.activations import relu, sigmoid
import random 
import time
from keras.layers import LeakyReLU
import h5py


from keras import Sequential
from keras.layers import Conv2D, Flatten, Dense
#import shap

#### creating a 3 hidden layer NN models with data

In [ ]:
cnn = Sequential()
#--------------First Hidden Layer----------------------------#
cnn.add(Dense(units = 8, activation='relu', kernel_initializer='he_uniform',input_dim= len(X_train.columns)))

#--------------Second  Hidden Layer -------------------------#
cnn.add(Dense(4, activation='relu', kernel_initializer='he_uniform'))

#--------------Third Hidden Layer -------------------------#
cnn.add(Dense(2, activation='relu', kernel_initializer='he_uniform'))

#--------------Output Layer----------------------------------#
cnn.add(Dense(1, activation='sigmoid', kernel_initializer='random_normal'))

#--------------Compiling the neural network------------------#
cnn.compile(optimizer ='adam',loss='binary_crossentropy', metrics =['accuracy'])

model = cnn.fit(X_train,y_train, batch_size=100, epochs=1000)

In [ ]:
cnn_predict = pd.DataFrame(cnn.predict(X_test))
cnn_predict

In [ ]:
y_test = y_test.reset_index()

In [ ]:
final_pred_cnn = pd.concat([cnn_predict,y_test],axis =1)

final_pred_cnn.columns =['Prob_1','level_0','index','return']
final_pred_cnn

## Creating the predcited column in the data using probablity cut off

In [ ]:
#final_pred.columns =['Prob_1','index','return']
final_pred_cnn['Predicted_cnn'] = np.where(final_pred_cnn['Prob_1']>=0.65,1,0)
final_pred_cnn.groupby(['return']).agg(Predicted_mean=('Predicted_cnn', 'mean'),Predicted_1=('Predicted_cnn', 'sum'),Predicted_all=('Predicted_cnn', 'count'))

In [ ]:
Accuracy_table =final_pred_cnn.groupby(['return']).agg(Predicted_mean=('Predicted_cnn', 'mean'),Predicted_1=('Predicted_cnn', 'sum'),Predicted_0=('Predicted_cnn', 'count'))
Accuracy_table= Accuracy_table.reset_index()
Accuracy_table.columns = ['Actual_return','predicted_mean','predicted_1','predicted_all']
Accuracy_table['predicted_0'] = Accuracy_table['predicted_all'] - Accuracy_table['predicted_1']
Accuracy = (Accuracy_table['predicted_0'][0] + Accuracy_table['predicted_1'][1] )/Accuracy_table['predicted_all'].sum()
Accuracy

In [ ]:
final_pred_cnn

## Creating the confusion matrix table to asses the model perfromance and create lift chart

In [ ]:

p_table =pd.DataFrame()

list_a = [10,20,30,40,50,60,70,80,90]

i=0

for n in list_a:
           

            p_table.loc[i,'Percentile'] = n

            p_table.loc[i,'Cut_Off_Probability'] = np.percentile(np.array(final_pred_cnn['Prob_1']),n)

            cut_off = p_table.loc[i,'Cut_Off_Probability']

            p_table.loc[i,'TN'] = len(final_pred_cnn[final_pred_cnn['return'] == 0]\
                                      [final_pred_cnn['Prob_1'] < cut_off])

            p_table.loc[i,'FP'] = len(final_pred_cnn[final_pred_cnn['return'] == 0]\
                                      [final_pred_cnn['Prob_1'] >= cut_off])

            p_table.loc[i,'FN'] = len(final_pred_cnn[final_pred_cnn['return'] == 1]\
                                      [final_pred_cnn['Prob_1'] < cut_off])

            p_table.loc[i,'TP'] = len(final_pred_cnn[final_pred_cnn['return'] == 1]\
                                      [final_pred_cnn['Prob_1'] >= cut_off])

            p_table.loc[i,'Total'] = p_table.loc[i,'TN'] + p_table.loc[i,'FP'] + p_table.loc[i,'FN'] + p_table.loc[i,'TP']

            p_table.loc[i,'FP+TP'] = p_table.loc[i,'FP'] + p_table.loc[i,'TP']

        

            p_table.loc[i,'Predicted 1 Accuracy'] = p_table.loc[i,'TP']/(p_table.loc[i,'TP'] + p_table.loc[i,'FP'])

          

            i+=1



p_table.to_csv("Cnn_lift_chart.csv")


#### NN Models are not perfroming and will need to be iterated further for improvments

# Section 4  - Prediction of the Validation data 

In [ ]:
#### create some derived varibale in the data 

df_validation = pd.read_csv(path + "TestingData_For_Candidate.csv")

df_validation['delivery_date']=pd.to_datetime(df_validation['delivery_date'],format = '%m/%d/%Y')
df_validation['order_date']=pd.to_datetime(df_validation['order_date'],format = '%m/%d/%Y')
df_validation['user_reg_date']=pd.to_datetime(df_validation['user_reg_date'],format = '%m/%d/%Y')
df_validation['user_dob']=pd.to_datetime(df_validation['user_dob'],format = '%m/%d/%Y')
df_validation['Today'] =pd.to_datetime(date.today(),format = '%Y-%m-%d')


df_validation['Delivery_Time'] = (df_validation['delivery_date'] -df_validation['order_date'])// timedelta(days=1)
df_validation['User_Age_years'] = (df_validation['Today'] - df_validation['user_dob'])// timedelta(days=365.2425)
df_validation['System_Age_months'] =(df_validation['Today'] - df_validation['user_reg_date'])// timedelta(days=30)

In [ ]:
df_validation['Delivery_Time'] = np.where(df_validation['Delivery_Time']< 0, df_validation['Delivery_Time'].mean(),df_validation['Delivery_Time'] )

In [ ]:
df_validation['Delivery_Time'] = df_validation['Delivery_Time'].fillna(df_validation['Delivery_Time'].mean())
df_validation['Delivery_Time']

##### Addition of variables using the historical return rates to the validation dataset

In [ ]:
##### Addtiion of User Return History

test = df[['user_return_history','order_date','user_id']]
test.drop_duplicates(keep = False, inplace = True)
test2 =  df_validation[['order_date','user_id']]
test2.drop_duplicates(keep = False, inplace = True)
test = test.append(test2)
test = test.sort_values(by=['user_id','order_date'])
test.drop_duplicates(keep = False, inplace = True)
test['user_return_history'] = test['user_return_history'].ffill()
test =test.drop_duplicates(subset=['order_date', 'user_id'])
df_validation = df_validation.merge(test, how ='left', on=['user_id','order_date'])
df_validation['user_return_history']= df_validation['user_return_history'].fillna(0)


In [ ]:
df_validation

In [ ]:
df_validation['user_return_history'].describe()

#### adding the other derived trend variables using the traning data 

In [ ]:
variable_list = ['brand_id','item_id','item_size','user_state','item_color']

for each in variable_list :
    
    var = str(each) + "_return_trend"
    
    test = df[['order_date',var,each]]
    
    
    test.drop_duplicates(keep = False, inplace = True)
    
    test2 =  df_validation[['order_date',each]]
    
    test2.drop_duplicates(keep = False, inplace = True)
    
    
    test = test.append(test2)
    
    test = test.sort_values(by=[each,'order_date'])
    
    
    test.drop_duplicates(keep = False, inplace = True)
    
    test[var] = test[var].ffill()
    
    test =test.drop_duplicates(subset=['order_date', each])
    
    
    df_validation = df_validation.merge(test, how ='left', on=[each,'order_date'])
    df_validation[var]= df_validation[var].fillna(0)


df_validation ['Price'] = df_validation ['item_price'] 

#### ADDING THE DUMMY VARIABLES

In [ ]:
df_validation_updated =pd.get_dummies(df_validation, columns=["item_id","item_size","item_color","brand_id","user_title","user_state"])

In [ ]:
df_validation_updated

In [ ]:
df_validation['user_return_history']

In [ ]:
df_validation_updated['type'] = 0
df_updated['type'] = 1
df_overall= df_updated.append(df_validation_updated)

In [ ]:
df_overall = df_overall.fillna(0)

In [ ]:
df_overall.columns[11:]

#### Using the overall data for standarization

In [ ]:
sc = StandardScaler()
X_validation =  pd.DataFrame(sc.fit_transform(df_overall.iloc[:,11:]))

In [ ]:
X_validation.columns =df_overall.iloc[:,11:].columns

In [ ]:
X_validation = X_validation[X_validation['type']<0]

In [ ]:
X_validation = X_validation.reset_index()

In [ ]:
X_validation.columns

In [ ]:
X.columns

In [ ]:
X_validation=X_validation[X.columns]
X_validation

# prediction of the valdiation set using the Random Forrest Models

In [ ]:
rfc_predict = pd.DataFrame(model.predict_proba(X_validation))
rfc_predict

In [ ]:

rfc_predict.columns =['Prob_0','Prob_1']

final_pred_validation =pd.DataFrame()


final_pred_validation['Predicted'] = np.where(rfc_predict['Prob_1']>=0.48,1,0)

In [ ]:
final_pred_validation['Predicted'].sum()

In [ ]:
final_pred_validation = pd.concat([final_pred_validation,df_validation],axis =1)
final_pred_validation

In [ ]:
final_pred_validation.to_csv("Final_prediction.csv")